In [ ]:
# ==============================================================
# 01_EDA – Multi-Country Intelligent Data Agent
# Title: A Multi-Agent Deep Reinforcement Learning Framework with
#        CNN-Encoded Alternative Data for Real-Time Credit Decisioning
#        in Emerging Markets
# ==============================================================

import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
%matplotlib inline

# --------------------------------------------------------------
# Paths
# --------------------------------------------------------------
ROOT = Path(".")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_SYNTHETIC = ROOT / "data" / "synthetic"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_SYNTHETIC.mkdir(parents=True, exist_ok=True)

print("Multi-Country Credit Data Agent initialized...")

# --------------------------------------------------------------
# 1. Load Original German Credit Dataset
# --------------------------------------------------------------
col_names = [
    'checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount',
    'savings', 'employment', 'installment_rate', 'personal_status_sex',
    'other_debtors', 'residence_since', 'property', 'age',
    'other_installment', 'housing', 'existing_credits', 'job',
    'num_dependents', 'telephone', 'foreign_worker', 'target'
]

df_german = pd.read_csv(DATA_RAW / "german.data", sep=' ', header=None, names=col_names)
df_german['default'] = (df_german['target'] == 2).astype(int)

print(f"Original German Credit shape: {df_german.shape}")
print(f"Original default rate: {df_german['default'].mean():.2%}")

# --------------------------------------------------------------
# 2. Intelligent Thin-file Proxy (Emerging Market style)
# --------------------------------------------------------------
df_german['thin_file'] = (
    ((df_german['checking_status'] == 'A14') | (df_german['savings'].isin(['A61', 'A65']))) &
    (df_german['employment'].isin(['A71', 'A72']))
).astype(int)

print(f"Thin-file rate (German): {df_german['thin_file'].mean():.2%}")
print(f"Default rate – Thin-file: {df_german.loc[df_german['thin_file']==1, 'default'].mean():.2%}")
print(f"Default rate – Thick-file: {df_german.loc[df_german['thin_file']==0, 'default'].mean():.2%}")

# --------------------------------------------------------------
# 3. Multi-Country Generation Agent
#    Creates realistic versions for Emerging Markets + USA/Europe
# --------------------------------------------------------------
def generate_country_version(base_df: pd.DataFrame, country: str, n_samples: int = 8000, seed: int = 42):
    """
    Intelligent agent that transforms German Credit into country-specific
    emerging market / global versions while preserving core risk structure.
    """
    rng = np.random.default_rng(seed)
    df = base_df.sample(n=min(n_samples, len(base_df)), replace=True, random_state=seed).reset_index(drop=True)

    # Country-specific adjustment factors (calibrated for research realism)
    country_params = {
        "INDIA":   {"default_boost": 1.25, "thin_boost": 1.90, "income_scale": 0.22, "loan_scale": 0.28, "mobile_intensity": 0.75},
        "NIGERIA": {"default_boost": 1.40, "thin_boost": 2.10, "income_scale": 0.18, "loan_scale": 0.22, "mobile_intensity": 0.82},
        "KENYA":   {"default_boost": 1.30, "thin_boost": 1.85, "income_scale": 0.20, "loan_scale": 0.25, "mobile_intensity": 0.88},
        "INDONESIA":{"default_boost": 1.15, "thin_boost": 1.70, "income_scale": 0.25, "loan_scale": 0.30, "mobile_intensity": 0.70},
        "BRAZIL":  {"default_boost": 1.20, "thin_boost": 1.55, "income_scale": 0.35, "loan_scale": 0.40, "mobile_intensity": 0.60},
        "CHINA":   {"default_boost": 0.85, "thin_boost": 1.45, "income_scale": 0.45, "loan_scale": 0.50, "mobile_intensity": 0.80},
        "USA":     {"default_boost": 0.95, "thin_boost": 0.90, "income_scale": 1.15, "loan_scale": 1.20, "mobile_intensity": 0.35},
        "EUROPE":  {"default_boost": 0.80, "thin_boost": 0.75, "income_scale": 0.90, "loan_scale": 0.95, "mobile_intensity": 0.40},
    }

    p = country_params.get(country, country_params["INDIA"])

    # Adjust thin-file prevalence
    thin_prob = np.clip(df['thin_file'].mean() * p["thin_boost"], 0.15, 0.60)
    df['thin_file'] = (rng.random(len(df)) < thin_prob).astype(int)

    # Adjust default probability (keep rank-order but shift rate)
    base_default_rate = df['default'].mean()
    target_default = np.clip(base_default_rate * p["default_boost"], 0.08, 0.32)

    # Simple probabilistic adjustment
    noise = rng.normal(0, 0.08, len(df))
    risk_score = df['default'] * 0.7 + df['thin_file'] * 0.3 + noise
    threshold = np.percentile(risk_score, (1 - target_default) * 100)
    df['default'] = (risk_score > threshold).astype(int)

    # Scale financial amounts
    df['credit_amount'] = (df['credit_amount'] * p["loan_scale"] * rng.uniform(0.85, 1.15, len(df))).round(0)
    df['income_proxy'] = (df['credit_amount'] * rng.uniform(2.5, 6.5, len(df)) * p["income_scale"] * 20).round(0)

    # ----------------------------------------------------------
    # Generate Synthetic Sequential Alternative Data (for CNN)
    # 30 timesteps × 8 channels  →  perfect for RQ2
    # ----------------------------------------------------------
    SEQ_LEN = 30
    N_CHANNELS = 8
    txn_seq = rng.normal(0, 0.35, size=(len(df), SEQ_LEN, N_CHANNELS)).astype(np.float32)

    for i in range(len(df)):
        if df.loc[i, 'default'] == 1:
            # Deteriorating pattern for bad credit
            txn_seq[i, -12:, 0] -= np.linspace(0.2, 0.9, 12)      # declining inflows
            txn_seq[i, -12:, 1] += np.linspace(0.15, 0.7, 12)     # rising outflows
            txn_seq[i, :, 2] += 0.35                               # higher volatility
        else:
            # Stable / improving pattern
            txn_seq[i, -12:, 0] += np.linspace(0.1, 0.45, 12)
            txn_seq[i, :, 3] += 0.25

        # Country-specific mobile money intensity
        txn_seq[i, :, 4] += p["mobile_intensity"] * rng.uniform(0.6, 1.2)

    df['country'] = country
    return df, txn_seq

# --------------------------------------------------------------
# 4. Generate Multi-Country Portfolio (Emerging Markets focus)
# --------------------------------------------------------------
target_countries = ["INDIA", "NIGERIA", "KENYA", "INDONESIA", "BRAZIL", "CHINA", "USA", "EUROPE"]

all_dfs = []
all_seqs = []

print("\n=== Multi-Country Generation Agent Running ===")
for idx, country in enumerate(target_countries):
    df_c, seq_c = generate_country_version(df_german, country, n_samples=7000, seed=42+idx)
    all_dfs.append(df_c)
    all_seqs.append(seq_c)

    print(f"✓ {country:12} | Samples: {len(df_c):,} | Default: {df_c['default'].mean():.2%} | "
          f"Thin-file: {df_c['thin_file'].mean():.2%}")

df_global = pd.concat(all_dfs, ignore_index=True)
txn_global = np.concatenate(all_seqs, axis=0)

print(f"\nGlobal dataset shape: {df_global.shape}")
print(f"Transaction sequences shape: {txn_global.shape}")

# --------------------------------------------------------------
# 5. Key Visualizations for Research Questions
# --------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# RQ1 / RQ4 – Default & Thin-file by country
order = df_global.groupby("country")["default"].mean().sort_values(ascending=False).index
sns.barplot(data=df_global, x="country", y="default", order=order, ax=axes[0,0], palette="Reds_r")
axes[0,0].set_title("Default Rate by Country (RQ1 & RQ4)")
axes[0,0].tick_params(axis='x', rotation=45)

sns.barplot(data=df_global, x="country", y="thin_file", order=order, ax=axes[0,1], palette="Blues_r")
axes[0,1].set_title("Thin-file Prevalence by Country (Emerging Markets Focus)")
axes[0,1].tick_params(axis='x', rotation=45)

# Thin-file vs Default
sns.boxplot(data=df_global, x="thin_file", y="credit_amount", hue="default", ax=axes[1,0])
axes[1,0].set_title("Credit Amount by Thin-file & Default Status")
axes[1,0].set_xticklabels(["Thick-file", "Thin-file"])

# Sequence signal preview (RQ2)
mean_good = txn_global[df_global['default']==0].mean(axis=0)
mean_bad  = txn_global[df_global['default']==1].mean(axis=0)
sns.heatmap(mean_bad.T - mean_good.T, ax=axes[1,1], cmap="RdBu_r", center=0)
axes[1,1].set_title("Alternative Data Signal Difference (Bad - Good)\nCNN will learn these patterns (RQ2)")
axes[1,1].set_xlabel("Day")
axes[1,1].set_ylabel("Channel")

plt.tight_layout()
plt.savefig(DATA_PROCESSED / "eda_multi_country_overview.png", dpi=140, bbox_inches="tight")
plt.show()

# --------------------------------------------------------------
# 6. Save all artifacts
# --------------------------------------------------------------
df_global.to_csv(DATA_SYNTHETIC / "global_credit_from_german.csv", index=False)
np.save(DATA_SYNTHETIC / "global_transaction_sequences.npy", txn_global)

# Also save original processed German version
df_german.to_csv(DATA_PROCESSED / "german_credit_processed.csv", index=False)

print("\n=== Files Saved ===")
print("• data/synthetic/global_credit_from_german.csv")
print("• data/synthetic/global_transaction_sequences.npy")
print("• data/processed/german_credit_processed.csv")
print("• data/processed/eda_multi_country_overview.png")
print("\n✅ Multi-Country EDA Agent completed successfully.")
print("This dataset now supports RQ1 (MARL), RQ2 (CNN alternative data), RQ3 (adaptation), RQ4 (inclusion & profitability).")